# Task 6 · Base vs SFT · Simple zero-shot CoT
Clone the shared semeval repository and install the package in `task 6/`.
No code ZIP or local dataset upload. Edit REPO_URL / GIT_REF in the cell below.
Use Runtime → Change runtime type → GPU for training. The transfer notebook can run on CPU.


In [ ]:
# Repository settings: editable in the notebook.
REPO_URL = 'https://github.com/iamtro-2006/semeval.git'
GIT_REF = 'main'  # branch, tag, or commit
CHECKOUT = '/content/semeval'
PROJECT_SUBDIR = 'task 6'

from pathlib import Path
import os, subprocess, sys
import codecs

def stream_run(cmd, check=True, cwd=None):
    # Route child output through IPython stdout, rather than its hidden OS file descriptor.
    env = dict(os.environ, PYTHONUNBUFFERED='1')
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env,
                               cwd=cwd if cwd is not None else globals().get('PROJECT_ROOT'))
    decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
    try:
        while True:
            chunk = os.read(process.stdout.fileno(), 4096)
            if not chunk:
                break
            sys.stdout.write(decoder.decode(chunk))
            sys.stdout.flush()
        sys.stdout.write(decoder.decode(b'', final=True))
        sys.stdout.flush()
        result = subprocess.CompletedProcess(cmd, process.wait())
    except BaseException:
        process.terminate()
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    finally:
        process.stdout.close()
    if check:
        result.check_returncode()
    return result

checkout = Path(CHECKOUT).expanduser()
if not checkout.is_absolute():
    raise ValueError('CHECKOUT must be absolute, e.g. /content/semeval')
checkout = checkout.resolve()
if not checkout.exists():
    checkout.parent.mkdir(parents=True, exist_ok=True)
    stream_run(['git', 'clone', REPO_URL, str(checkout)], check=True, cwd=checkout.parent)
else:
    actual = subprocess.check_output(['git', '-C', str(checkout), 'remote', 'get-url', 'origin'], text=True).strip()
    if actual != REPO_URL:
        raise ValueError('CHECKOUT belongs to a different repository; choose another CHECKOUT')
    dirty = subprocess.check_output(['git', '-C', str(checkout), 'status', '--porcelain'], text=True)
    if dirty.strip():
        raise RuntimeError('Checkout has edits; preserve your changes or choose a fresh CHECKOUT before switching revision')
stream_run(['git', '-C', str(checkout), 'fetch', 'origin', GIT_REF], check=True, cwd=checkout.parent)
stream_run(['git', '-C', str(checkout), 'checkout', '--detach', 'FETCH_HEAD'], check=True, cwd=checkout.parent)
PROJECT_ROOT = (checkout / PROJECT_SUBDIR).resolve()
SRC = PROJECT_ROOT / 'src'
if not (SRC / 'mmcultureqa/__init__.py').is_file():
    raise FileNotFoundError(f'Project package not found: {SRC / "mmcultureqa"}')
os.chdir(PROJECT_ROOT)
# Editable installation alone does not refresh sys.path in an already running kernel.
# Export PYTHONPATH too, so child Python processes and workers use this same checkout.
sys.path[:] = [str(SRC)] + [p for p in sys.path if p != str(SRC)]
python_paths = [p for p in os.environ.get('PYTHONPATH', '').split(os.pathsep) if p and p != str(SRC)]
os.environ['PYTHONPATH'] = os.pathsep.join([str(SRC), *python_paths])
import importlib
importlib.invalidate_caches()
for name in list(sys.modules):
    if name == 'mmcultureqa' or name.startswith('mmcultureqa.'):
        del sys.modules[name]
stream_run([sys.executable, '-m', 'pip', 'install', '--no-deps', '--no-build-isolation', '-e', str(PROJECT_ROOT)], check=True)
from mmcultureqa.common import ROOT as PACKAGE_ROOT
if PACKAGE_ROOT.resolve() != PROJECT_ROOT:
    raise ImportError(f'mmcultureqa loaded from {PACKAGE_ROOT}, expected {PROJECT_ROOT}')
stream_run([sys.executable, '-c', 'import mmcultureqa; print("Child package:", mmcultureqa.__file__)'])
print('Project:', Path.cwd())
print('Commit:', subprocess.check_output(['git', '-C', str(checkout), 'rev-parse', 'HEAD'], text=True).strip())


## Install GPU environment
Restart the runtime if requested, then rerun clone/setup. GPU training has not been run on the CPU-only development machine.


In [ ]:
INSTALL_VLLM = True  # final generation uses vLLM after training exits
requirements = 'requirements-vllm.txt' if INSTALL_VLLM else 'requirements.txt'
stream_run([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT_ROOT / requirements)])
# Colab may have unrelated preinstalled-package conflicts; print the full report.
stream_run([sys.executable, '-m', 'pip', 'check'], check=False)
stream_run(['nvidia-smi'])


In [ ]:
# Fail early on dependency/import errors before downloading data or starting training.
import subprocess, sys
stream_run([sys.executable, '-c', """
import sys
from importlib.metadata import version
for name in ('numpy', 'torch', 'transformers', 'peft', 'trl', 'ms-swift', 'bitsandbytes', 'msgspec'):
    print(name, version(name), flush=True)
from transformers.modeling_utils import PreTrainedModel
from peft import PeftModel
from swift.llm import TrainArguments, RLHFArguments
import qwen_vl_utils
import decord
from bitsandbytes.nn import Linear4bit
from swift.llm import PtEngine
import mmcultureqa.common
import mmcultureqa.data.prepare
import mmcultureqa.training.sft
import mmcultureqa.training.grpo
import mmcultureqa.training.curves
import mmcultureqa.inference.predict
import mmcultureqa.evaluate
from swift.llm.train.sft import SwiftSft
from swift.llm.train.rlhf import SwiftRLHF
# Import the concrete trainer too: SwiftRLHF alone leaves its dependencies lazy.
from swift.trainers.rlhf_trainer.grpo_trainer import GRPOTrainer
from msgspec import field
from swift.llm import InferRequest, RequestConfig
if sys.argv[1] == '1':
    print('vllm', version('vllm'), flush=True)
    from vllm import LLM
    from swift.llm import VllmEngine
from swift.plugin import metric_mapping, orms, ORM
from swift.utils import Serializer
print('Training imports OK', flush=True)
""", '1' if INSTALL_VLLM else '0'])


## MMCQA folder and authentication
Only ROOT_FOLDER_ID selects storage. Paste the ID or URL of your MMCQA folder.
RCLONE_CONFIG_PATH points to your authenticated rclone.conf; this is not a storage root.
All data, adapters, checkpoints and results are addressed relative to the selected ID.
If your config is on mounted Drive, run the optional mount cell first.


In [ ]:
# Optional, only to access a config file stored on Drive:
# from google.colab import drive
# drive.mount('/content/drive')


In [ ]:
ROOT_FOLDER_ID = '1kcdByZiSJzGSoEtY9q0OEhsZwiTZkxpV'  # ID or URL of the MMCQA folder
RCLONE_CONFIG_PATH = '/content/rclone.conf'
REMOTE = 'iamtro'  # Google Drive remote in rclone.conf


In [ ]:
stream_run(['sudo', 'apt-get', '-qq', 'update'])
stream_run(['sudo', 'apt-get', '-qq', 'install', '-y', 'rclone'])
import configparser, re, tempfile
from urllib.parse import urlparse
config_source = Path(RCLONE_CONFIG_PATH).expanduser()
if not config_source.is_absolute():
    raise ValueError('RCLONE_CONFIG_PATH must be absolute, e.g. /content/rclone.conf')
if not config_source.is_file():
    raise FileNotFoundError(f'Upload your authenticated rclone.conf or mount Drive: {config_source}')
sections = configparser.RawConfigParser()
sections.read(config_source, encoding='utf-8-sig')
remotes = [name for name in sections.sections() if sections[name].get('type') == 'drive']
if not REMOTE:
    if len(remotes) != 1:
        raise ValueError(f'Select REMOTE from Google Drive sections: {remotes}')
    REMOTE = remotes[0]
if REMOTE not in remotes:
    raise ValueError('REMOTE must be a Google Drive section')
root_id = ROOT_FOLDER_ID.strip()
if root_id.startswith(('https://', 'http://')):
    match = re.search(r'/folders/([^/?]+)', urlparse(root_id).path)
    root_id = match.group(1) if match else ''
if not re.fullmatch(r'[A-Za-z0-9_-]+', root_id):
    raise ValueError('Enter the ID or folder URL of MMCQA in ROOT_FOLDER_ID')
temporary_config = tempfile.TemporaryDirectory(prefix='mmcqa-rclone-')
config_path = Path(temporary_config.name) / 'rclone.conf'
config_path.write_bytes(config_source.read_bytes())
config_path.chmod(0o600)
rclone_base = ['rclone', '--config', str(config_path), '--drive-root-folder-id', root_id]
def cloud(relative):
    if relative.startswith('/') or '..' in Path(relative).parts:
        raise ValueError('Storage paths must be relative to MMCQA')
    return f'{REMOTE}:{relative}'
def copy(source, destination, *extra):
    stream_run(rclone_base + ['copy', str(source), str(destination),
                   '--transfers', '4', '--checkers', '8', '--progress', *extra], check=True)
stream_run(rclone_base + ['lsd', cloud(''), '--max-depth', '1'], check=True)
print('All storage is relative to MMCQA folder ID:', root_id)


## Prepare data on Colab SSD
The download location is automatically derived from ROOT_FOLDER_ID.


In [ ]:
USE_DRIVE_DATA = True
if USE_DRIVE_DATA:
    copy(cloud('task6-data/raw'), PROJECT_ROOT / 'data/raw')
else:
    stream_run([sys.executable, '-m', 'mmcultureqa.data.prepare', 'download'], check=True)
stream_run([sys.executable, '-m', 'mmcultureqa.data.prepare', 'prepare'], check=True)


## Base FP16 vs saved SFT · inference only
One notebook, two sequential inference processes: original Qwen3-VL-4B-Instruct and your Drive best_model.
Both run in FP16 with the same simple zero-shot CoT prompt, image limits and decoding settings.
The merged SFT files on Drive remain BF16; FP16 casting happens at load time only.
No training or merging. Model processes exit before scoring or loading the next model.
Default: all 1,000 held-out validation questions. LIMIT=4 is an explicit inference smoke test.
Each malformed answer retries once; a second failure raises an error and preserves traces.


In [ ]:
import copy as copy_module
import json, re, yaml
from mmcultureqa.common import merge, read_jsonl, write_jsonl
from mmcultureqa.models.artifacts import require_merged_model
from mmcultureqa.storage import upload_filters

RUN_NAME = 'base_fp16_vs_sft_cot01'  # use a new name when changing settings
SFT_RUN_NAME = 'experiment03'
SFT_MODEL_STAGE = 'sft'
SPLIT = 'validation'  # validation for metrics; devtest/test for unlabelled inference
LIMIT = None
MAX_NEW_TOKENS = 1024
for name in (RUN_NAME, SFT_RUN_NAME):
    if not re.fullmatch(r'[A-Za-z0-9_-]+', name):
        raise ValueError('Use letters, digits, underscores or hyphens for run names')
if SFT_MODEL_STAGE not in {'sft', 'sft-smoke'} or SPLIT not in {'validation', 'devtest', 'test'}:
    raise ValueError('Choose a valid SFT stage and held-out split')
if LIMIT is not None and (not isinstance(LIMIT, int) or LIMIT < 1):
    raise ValueError('LIMIT must be null or a positive integer')

OUTPUT = PROJECT_ROOT / 'outputs/inference_runs' / RUN_NAME
OUTPUT.mkdir(parents=True, exist_ok=True)
local_sft = PROJECT_ROOT / 'outputs/inputs/models' / SFT_RUN_NAME / SFT_MODEL_STAGE / 'best_model'
copy(cloud(f'task6-runs/{SFT_RUN_NAME}/{SFT_MODEL_STAGE}/best_model'), local_sft)
require_merged_model(local_sft)
saved_sft = json.loads((local_sft / 'run_config.json').read_text(encoding='utf-8'))
source = yaml.safe_load((PROJECT_ROOT / 'configs/sft.yaml').read_text(encoding='utf-8'))
inference = yaml.safe_load((PROJECT_ROOT / 'configs/inference.yaml').read_text(encoding='utf-8'))
inference.pop('extends', None)
template = merge(source, inference)
for key in ('min_pixels', 'max_pixels'):
    template['model'][key] = saved_sft['model'][key]
rows = read_jsonl(PROJECT_ROOT / 'data/prepared' / f'{SPLIT}.jsonl')
if SPLIT == 'validation':
    train = read_jsonl(PROJECT_ROOT / 'data/prepared/train.jsonl')
    if ({r['id'] for r in rows} & {r['id'] for r in train}
            or {r['image'] for r in rows} & {r['image'] for r in train}):
        raise ValueError('Training/validation overlap')
rows = rows[:LIMIT] if LIMIT is not None else rows
if not rows:
    raise ValueError('Selected split is empty')
gold = OUTPUT / 'gold_en.jsonl'
write_jsonl(gold, rows)
settings = {'source_sft_run': SFT_RUN_NAME, 'source_sft_stage': SFT_MODEL_STAGE, 'split': SPLIT,
            'limit': LIMIT, 'count': len(rows), 'dtype': 'float16',
            'system_prompts': {'base': source.get('system_prompt'), 'sft': saved_sft.get('system_prompt')},
            'prompt': (PROJECT_ROOT / 'prompts/qa_cot_simple.txt').read_text(encoding='utf-8'),
            'seed': 42, 'temperature': 0.7, 'top_p': 0.8, 'top_k': 20, 'presence_penalty': 1.5,
            'max_new_tokens': MAX_NEW_TOKENS}
(OUTPUT / 'settings.json').write_text(json.dumps(settings, ensure_ascii=False, indent=2), encoding='utf-8')

def upload_inference_comparison():
    copy(OUTPUT, cloud(f'task6-inference/{RUN_NAME}'), *upload_filters('inference_comparison'))


## Run both models on the same questions
Original base loads from its pinned Hugging Face revision. SFT loads from the local Drive download.
Primary decoding uses sampling temperature 0.7, top-p 0.8, top-k 20, presence penalty 1.5.
These are controlled inference settings, not proof of a quality improvement. Inspect the raw traces.


In [ ]:
for label in ('base', 'sft'):
    folder = OUTPUT / label
    folder.mkdir(parents=True, exist_ok=True)
    cfg = copy_module.deepcopy(template)
    cfg['model'].update(name=source['model']['name'] if label == 'base' else str(local_sft),
        revision=source['model']['revision'] if label == 'base' else None,
        adapter=None, adapter_revision=None, processor=None, processor_revision=None,
        load_in_4bit=False, dtype='float16', merged_model=label == 'sft', local_files_only=label == 'sft')
    cfg['data'].update(limit=LIMIT)
    cfg['training']['seed'] = 42
    cfg['inference'].update(backend='vllm', dtype='float16',
        split=SPLIT, output=str(folder / 'predictions.jsonl'), resume=True,
        prompt_mode='cot', prompt_file=str(PROJECT_ROOT / 'prompts/qa_cot_simple.txt'),
        max_new_tokens=MAX_NEW_TOKENS, vllm_max_model_len=4096, vllm_max_num_seqs=4,
        do_sample=True, temperature=0.7, top_p=0.8, top_k=20, presence_penalty=1.5,
        on_format_error='raise', format_retries=1)
    config_file = folder / 'config.json'
    config_file.write_text(json.dumps(cfg, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'Running {label.upper()} · FP16 · {len(rows)} questions', flush=True)
    result = stream_run([sys.executable, '-m', 'mmcultureqa.inference.predict', '--config', str(config_file)], check=False)
    upload_inference_comparison()
    result.check_returncode()
    predicted = read_jsonl(folder / 'predictions.jsonl')
    if len(predicted) != len(rows) or {r['id'] for r in predicted} != {r['id'] for r in rows}:
        raise ValueError(f'Incomplete predictions: {label}')
    if all(r.get('answer') for r in rows):
        # Inference subprocess has exited, releasing its model before loading the scorer.
        stream_run([sys.executable, '-m', 'mmcultureqa.score', '--dtype', 'float16',
            '--gold', str(gold), '--pred', str(folder / 'predictions.jsonl'), '--lang', 'en',
            '--out', str(folder / 'scores.json')])
        upload_inference_comparison()
    else:
        print(f'{SPLIT}: no public labels; predictions/traces saved without scores.')


## Model comparison
Same questions, prompt, preprocessing and FP16 inference; compare base vs SFT weights.
Metrics are shown on a 0–100 scale; Δ = SFT − base, in points.


In [ ]:
from IPython.display import display
import pandas as pd

if all((OUTPUT / label / 'scores.json').is_file() for label in ('base', 'sft')):
    metrics = ['BERTScore-F1', 'BERTScore-P', 'BERTScore-R', 'BLEU', 'ROUGE-L', 'Coverage']
    table_rows = []
    for label in ('base', 'sft'):
        score = json.loads((OUTPUT / label / 'scores.json').read_text(encoding='utf-8'))['submission_result']
        table_rows.append({key: score[key] * (1 if key == 'BLEU' else 100) for key in metrics})
    table = pd.DataFrame(table_rows, index=['Original base · FP16', 'SFT best · FP16'])
    table.loc['Δ SFT − base'] = table.iloc[1] - table.iloc[0]
    styled = table.style.format('{:.3f}').set_caption(f'Base vs SFT · Zero-shot CoT · {len(rows):,} questions').highlight_max(
        subset=pd.IndexSlice[table.index[:2], :], axis=0, color='#d1fae5').set_table_styles([
            {'selector': 'caption', 'props': [('font-size', '18px'), ('font-weight', 'bold'), ('padding', '12px')]},
            {'selector': 'th, td', 'props': [('padding', '10px 14px'), ('text-align', 'right')]}])
    display(styled)
    table.to_csv(OUTPUT / 'comparison.csv', encoding='utf-8-sig')
    (OUTPUT / 'comparison.html').write_text(styled.to_html(), encoding='utf-8')
    (OUTPUT / 'comparison.json').write_text(json.dumps({'settings': settings, 'metrics_0_100': table.to_dict(orient='index')},
        ensure_ascii=False, indent=2), encoding='utf-8')
    upload_inference_comparison()
else:
    print('No comparison scores: use the labelled validation split and complete both inference runs.')
